In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import pylab as pl
import numpy as np
%matplotlib inline

In [ ]:
Step 1:  Problem Definition      ✅ done!
Step 2:  Data Collection         ✅ done!
Step 3:  EDA                     → explore, understand, identify issues
Step 4:  Data Splitting          → Train/Test (stratify not needed, Regression)
Step 5:  Data Cleaning           → missing values, wrong types, outliers
Step 6:  Baseline Model          → always predict mean → reference point
Step 7:  Feature Engineering     → encoding, scaling, new features
Step 8:  Feature Selection       → drop useless/correlated features
Step 10: Train Models            → Linear, Random Forest, XGBoost...
Step 11: Hyperparameter Tuning   → tune best models
Step 12: Final Evaluation        → R² on test set → groupX.txt
         +
         Wait for Inference Data → predict → groupX.csv

In [ ]:
Target:  Predict Sales  →  a number  →  Regression! ✅
Metric:  R²  ← explicitly required in the challenge
Extra:   MAE/RMSE for better understanding

Output:  groupX.csv  ← all columns + sales prediction column
         groupX.txt  ← R² score from test set

In [5]:
df = pd.read_csv("sales.csv")
df.head()

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,425390,366,4,2013-04-18,517,1,0,0,0,4422
1,291687,394,6,2015-04-11,694,1,0,0,0,8297
2,411278,807,4,2013-08-29,970,1,1,0,0,9729
3,664714,802,2,2013-05-28,473,1,1,0,0,6513
4,540835,726,4,2013-10-10,1068,1,1,0,0,10882


In [6]:
df.describe()

,Unnamed: 0,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales
count,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000,640840.000000
mean,355990.675084,558.211348,4.000189,633.398577,0.830185,0.381718,0.178472,5777.469011
std,205536.290268,321.878521,1.996478,464.094416,0.375470,0.485808,0.382910,3851.338083
min,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,178075.750000,280.000000,2.000000,405.000000,1.000000,0.000000,0.000000,3731.000000
50%,355948.500000,558.000000,4.000000,609.000000,1.000000,0.000000,0.000000,5746.000000
75%,533959.250000,837.000000,6.000000,838.000000,1.000000,1.000000,0.000000,7860.000000
max,712044.000000,1115.000000,7.000000,5458.000000,1.000000,1.000000,1.000000,41551.000000


In [8]:
df.shape

(640840, 10)

In [9]:
df.dtypes

Unnamed: 0              int64
store_ID                int64
day_of_week             int64
date                   object
nb_customers_on_day     int64
open                    int64
promotion               int64
state_holiday          object
school_holiday          int64
sales                   int64
dtype: object

In [12]:
# Missing values
print("Missing values per column:")
print(df.isna().sum())
print()
print("Total missing:", df.isna().sum().sum())
print()

Missing values per column:
Unnamed: 0             0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
sales                  0
dtype: int64

Total missing: 0



In [13]:
# All rows where sales = 0
closed = df[df['sales'] == 0]

print("Total rows with sales=0:", len(closed))
print()
print("Are they all closed (open=0)?")
print(closed['open'].value_counts())
print()
print("Day of week distribution:")
print(closed['day_of_week'].value_counts().sort_index())
print()
print("Sample rows:")
closed.head(10)

Total rows with sales=0: 108855

Are they all closed (open=0)?
open
0    108824
1        31
Name: count, dtype: int64

Day of week distribution:
day_of_week
1     4517
2     1091
3     2379
4     7093
5     4564
6      449
7    88762
Name: count, dtype: int64

Sample rows:


,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
39,357935,1043,7,2013-08-18,0,0,0,0,0,0
44,366083,1093,7,2013-10-06,0,0,0,0,0,0
50,586075,864,7,2013-09-01,0,0,0,0,0,0
56,350680,929,7,2015-02-01,0,0,0,0,0,0
58,561112,532,7,2013-02-24,0,0,0,0,0,0


In [18]:
# Rows where open=1
open_stores = df[df['open'] == 1]

print("Total open rows:", len(open_stores))
print()
print("Sales stats for open stores:")
print(open_stores['sales'].describe())
print()
print("Any sales=0 when open=1?")
print((open_stores['sales'] == 0).sum())
print()
print("Sample:")
open_stores[['open', 'sales']].head(30)

Total open rows: 532016

Sales stats for open stores:
count    532016.000000
mean       6959.251679
std        3105.241710
min           0.000000
25%        4861.000000
50%        6372.000000
75%        8365.000000
max       41551.000000
Name: sales, dtype: float64

Any sales=0 when open=1?
31

Sample:


,open,sales
0,1,4422
1,1,8297
2,1,9729
3,1,6513
4,1,10882
5,1,8406
7,1,11162
8,1,5559
9,1,3997
11,1,6267


In [19]:
print("Open=1 rows:", (df['open'] == 1).sum())
print("Open=0 rows:", (df['open'] == 0).sum())
print("Total rows:", len(df))

Open=0: 108,824 rows  ← closed stores → drop
Open=1: 532,016 rows  ← open stores → keep
Total:  640,840 rows

Open=1 rows: 532016
Open=0 rows: 108824
Total rows: 640840


In [21]:
#unique score : 
print("Unique sales values:", df['sales'].nunique())
print()
print("Sales value counts (top 20):")
print(df['sales'].value_counts().head(20))
print()
print("How many times sales=0:")
print((df['sales'] == 0).sum())

Unique sales values: 20129

Sales value counts (top 20):
sales
0       108855
5674       146
6049       134
5449       130
5723       128
5584       127
6214       127
5044       126
5316       126
5931       126
5680       126
6171       126
4842       125
5460       123
5093       123
6439       122
5342       122
5553       121
5989       121
5194       121
Name: count, dtype: int64

How many times sales=0:
108855


In [22]:
print("Open=1 rows:", (df['open'] == 1).sum())
print("Open=0 rows:", (df['open'] == 0).sum())
print("Total rows:", len(df))

#Open=0: 108,824 rows  ← closed stores → drop!
#Open=1: 532,016 rows  ← open stores → keep!
#Total:  640,840 rows

#sales=0: 108,855 rows
#open=0:  108,824 rows
#108,824 → closed (open=0) AND sales=0  ✅ expected
#31      → open=1 BUT sales=0           ⚠️ anomaly!




Open=1 rows: 532016
Open=0 rows: 108824
Total rows: 640840


In [24]:
# Open but no sales
anomalies = df[(df['open'] == 1) & (df['sales'] == 0)]
print("Open=1 but sales=0:", len(anomalies))
print()
print(anomalies)

# Drop them → likely data errors ✅

Open=1 but sales=0: 31

        Unnamed: 0  store_ID  day_of_week        date  nb_customers_on_day  \
54379       392697       339            3  2013-01-30                    0   
61554       194070      1039            3  2013-07-10                    0   
63404       573822       983            5  2014-01-17                    0   
82644       540904       387            4  2014-07-24                    0   
117219      432135       762            4  2013-01-17                    0   
137903      236556       303            4  2014-07-24                    0   
163742      253274        25            3  2014-02-12                    0   
180912      502164       835            4  2014-09-11                    0   
188539      389729       835            3  2014-09-10                    0   
209801      581607       548            5  2014-09-05                    0   
235104       91884       623            6  2014-01-25                    0   
238863      143926      1017            

In [ ]:
# Step 3:  EDA → explore, understand, identify issues

    # Understanding the fEATURES :
    store_ID             → which store (1-1115)
    day_of_week          → 1-7
    date                 → date as string 
    nb_customers_on_day  → number of customers
    open                 → 0=closed, 1=open
    promotion            → 0/1
    state_holiday        → '0','a','b','c'
    school_holiday       → 0/1
    sales                → TARGET 🎯


    # Identified Issues 

    
    #Encoding / Convert
    date : string - datetime conversation
    state_holiday : needs encoding 

    #Drop issues (Before Spliting)
    sales = 0     → 108,855 rows where store is CLOSED (open=0)
        → remove these
    unamed Column should be removed
    No missing values ✅

    #Scales issues (after Spliting)
    
    Store Id / day_of_week.. scale ? 

In [ ]:
#Issues found:
1. Unnamed column    → useless index column
2. open=0 rows       → 108,824 closed store rows
3. sales=0, open=1   → 31 anomaly rows
4. date              → wrong type (string) + needs extraction
5. state_holiday     → categorical ('0','a','b','c') → needs encoding
6. open column       → constant after filtering → useless

#Action Plan before Splitting:

Step 1: Drop Unnamed column ✅
Step 2: Filter → keep only open=1 rows ✅
#Step 3: Drop the 31 anomaly rows (sales=0, open=1)
Step 4: Drop open column (constant now) ✅
Step 5: drop date ✅
Step 6: Encode state_holiday ('0','a','b','c' → numbers) --- Binary / Not One-Hot ✅
Step 7: Drop duplicates ✅


Step X — Spliting ✅


#After Cleaning → Split → Then: ✅

Step 7: Scaling  → fit ONLY on train! ✅
Step 8: Baseline → predict mean of train sales
Step 9: Train models → Linear, Random Forest, XGBoost
Step 10: Evaluate → R² on test set

In [25]:
#Step 1: Drop Unnamed column
df = df.drop(columns=['Unnamed: 0'])

# Verify
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (640840, 9)


In [28]:
# Script if the passed shop is clsed -> 0 , if oPEN -> USE OUR Model
# Step 2: Filter → keep only open=1 rows
# 
df = df[df['open'] == 1]
print("Shape after filtering:", df.shape)
print("Open values:", df['open'].value_counts())

Shape after filtering: (532016, 9)
Open values: open
1    532016
Name: count, dtype: int64


In [29]:
df.head()

,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,366,4,2013-04-18,517,1,0,0,0,4422
1,394,6,2015-04-11,694,1,0,0,0,8297
2,807,4,2013-08-29,970,1,1,0,0,9729
3,802,2,2013-05-28,473,1,1,0,0,6513
4,726,4,2013-10-10,1068,1,1,0,0,10882


In [32]:
df = df.drop(columns=['open'])

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (532016, 8)


In [33]:
# Drop date column
df = df.drop(columns=['date'])

# Verify
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

Columns: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday', 'sales']
Shape: (532016, 7)


In [34]:
# Check unique values
print("Unique values in state_holiday:")
print(df['state_holiday'].unique())
print()
print("Value counts:")
print(df['state_holiday'].value_counts())

Unique values in state_holiday:
['0' 'a' 'b' 'c']

Value counts:
state_holiday
0    531437
a       429
b       102
c        48
Name: count, dtype: int64


In [35]:
# Binary encoding: holiday or not
df['state_holiday'] = df['state_holiday'].apply(lambda x: 0 if x == '0' else 1)

# Verify
print("Unique values:", df['state_holiday'].unique())
print("Value counts:")
print(df['state_holiday'].value_counts())

Unique values: [0 1]
Value counts:
state_holiday
0    531437
1       579
Name: count, dtype: int64


In [36]:
df.dtypes

store_ID               int64
day_of_week            int64
nb_customers_on_day    int64
promotion              int64
state_holiday          int64
school_holiday         int64
sales                  int64
dtype: object

In [37]:
# Step 4 - Data Spliting 

from sklearn.model_selection import train_test_split

# Define features and target
X = df.drop(columns=['sales'])
y = df['sales']

# Step 1: Split off 30% for val+test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

# Step 2: Split 30% eq into val & test 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,    
    random_state=42
)

# Verify
print("X_train:", X_train.shape)  # ~70%
print("X_val:  ", X_val.shape)    # ~15%
print("X_test: ", X_test.shape)   # ~15%


X_train: (372411, 6)
X_val:   (79802, 6)
X_test:  (79803, 6)


In [38]:
#Baseline 
import numpy as np

# Baseline: always predict mean of training sales
baseline_pred = np.mean(y_train)

print(f"Baseline prediction (mean): {baseline_pred:.2f}")

# Evaluate on validation set
baseline_mae = np.mean(np.absolute(baseline_pred - y_val))
baseline_r2  = 1 - (np.sum((y_val - baseline_pred)**2) / 
                    np.sum((y_val - np.mean(y_val))**2))

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline R²:  {baseline_r2:.4f}")

Baseline prediction (mean): 6957.20
Baseline MAE: 2294.77
Baseline R²:  -0.0000


In [39]:
# Check duplicates
print("Total rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print()

# Check duplicates except target
print("Duplicate features (ignoring sales):")
print(df.duplicated(subset=df.columns.drop('sales')).sum())

Total rows: 532016
Duplicate rows: 68

Duplicate features (ignoring sales):
43061


In [ ]:
# Drop duplicates
df = df.drop_duplicates()

# Verify
print("Shape after dropping duplicates:", df.shape)
# Expected: 532016 - 68 = 531948 rows

In [40]:
# # Re-split with clean data
X = df.drop(columns=['sales'])
y = df['sales']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

X_train: (372411, 6)
X_val:   (79802, 6)
X_test:  (79803, 6)


In [41]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# fit ONLY on train!
scaler.fit(X_train[['store_ID']])

# What did scaler learn?
print(f"Mean learned: {scaler.mean_[0]:.2f}")
print(f"Std learned:  {scaler.scale_[0]:.2f}")

# Transform all sets with SAME scaler
X_train[['store_ID']] = scaler.transform(X_train[['store_ID']])
X_val[['store_ID']]   = scaler.transform(X_val[['store_ID']])
X_test[['store_ID']]  = scaler.transform(X_test[['store_ID']])

# Verify
print("\nBefore scaling: min=1, max=1115")
print(f"After scaling:")
print(X_train[['store_ID']].describe())



Mean learned: 557.97
Std learned:  321.64

Before scaling: min=1, max=1115
After scaling:
           store_ID
count  3.724110e+05
mean  -7.631813e-17
std    1.000001e+00
min   -1.731646e+00
25%   -8.642170e-01
50%    1.026779e-04
75%    8.644223e-01
max    1.731851e+00


In [42]:
# Make sure no sales-related columns leaked in X
print("X_train columns:")
print(X_train.columns.tolist())

# sales should NOT be in X!
assert 'sales' not in X_train.columns
print("✅ No data leakage detected!")

X_train columns:
['store_ID', 'day_of_week', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday']
✅ No data leakage detected!


In [ ]:
Data Leakage Checklist:
* Split before any Fitting 
* Fit() only on X_train
* transform() on all sets with same scaler 
* No Fututre Information in Features 
* Duplicates are removed before split


In [44]:
#Base Line

import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# Step 1: calculate mean from TRAINING only!
baseline_value = np.mean(y_train)
print(f"Baseline always predicts: {baseline_value:.2f}")

# Step 2: predict same value for every row in validation
baseline_predictions = np.full(
    shape=len(y_val),        # how many rows?
    fill_value=baseline_value # always same value!
)

print(f"First 5 predictions: {baseline_predictions[:5]}")
# → [5777, 5777, 5777, 5777, 5777]

# Step 3: evaluate on validation set
mae = mean_absolute_error(y_val, baseline_predictions)
r2  = r2_score(y_val, baseline_predictions)

print(f"\nBaseline MAE: {mae:.2f}")
print(f"Baseline R²:  {r2:.4f}")  # always 0!
print(f"\nOur model must beat:")
print(f"→ R²  > 0.0")
print(f"→ MAE < {mae:.2f}")

Baseline always predicts: 6957.20
First 5 predictions: [6957.19655166 6957.19655166 6957.19655166 6957.19655166 6957.19655166]

Baseline MAE: 2294.77
Baseline R²:  -0.0000

Our model must beat:
→ R²  > 0.0
→ MAE < 2294.77


In [46]:
# First Model Linear Regeression 

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ─────────────────────────────────────
# Model 1: Linear Regression
# ─────────────────────────────────────

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)

print("=" * 40)
print("1. Linear Regression")
print(f"   R²:  {r2_score(y_val, lr_pred):.4f}")
print(f"   MAE: {mean_absolute_error(y_val, lr_pred):.2f}")


1. Linear Regression
   R²:  0.7291
   MAE: 1156.53


In [47]:

# ─────────────────────────────────────
# Model 2: Random Forest
# ─────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("=" * 40)
print("2. Random Forest")
print(f"   R²:  {r2_score(y_val, rf_pred):.4f}")
print(f"   MAE: {mean_absolute_error(y_val, rf_pred):.2f}")

2. Random Forest
   R²:  0.9352
   MAE: 504.84


In [48]:
# ─────────────────────────────────────
# Model 3: XGBoost
# ─────────────────────────────────────
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)

print("=" * 40)
print("3. XGBoost")
print(f"   R²:  {r2_score(y_val, xgb_pred):.4f}")
print(f"   MAE: {mean_absolute_error(y_val, xgb_pred):.2f}")

3. XGBoost
   R²:  0.8655
   MAE: 829.39


In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Random Forest — Before Tuning
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("--- Random Forest Before Tuning ---")
print(f"R²:  {r2_score(y_val, rf_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_val, rf_pred):.2f}")

--- Random Forest Before Tuning ---
R²:  0.9352
MAE: 504.84


In [53]:
# Tuning the Random Forest model 

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# ─────────────────────────────────────
# Random Forest — Before Tuning:
# n_estimators=100  → number of trees (fixed at 100)
# max_depth=None    → unlimited depth → risk of overfitting!
# min_samples_split=2 → splits almost always → risk of overfitting!
# R²: 0.9352  MAE: 504.84


#Parameter      Before          Jetzt
#──────────────────────────────────────
#n_estimators   [200,300,500]   [200,300]    ← weniger
#max_depth      [10,20,30]      [10,20]      ← weniger
#min_samples    [5,10,20]       [5,10]       ← weniger
#min_samples_leaf [1,2,4]       ← komplett entfernt!
#max_features   ['sqrt','log2'] ← komplett entfernt!
# ─────────────────────────────────────

from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    'n_estimators':      [200, 300],
    'max_depth':         [10, 20],
    'min_samples_split': [5, 10],
}

rf_tuned = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    n_iter=5,       # only 5 combinations
    cv=2,           # only 2 folds
    scoring='r2',
    n_jobs=1,       # only 1 core → safe!
    random_state=42,
    verbose=1
)

rf_tuned.fit(X_train, y_train)

print("Best params:", rf_tuned.best_params_)

rf_tuned_pred = rf_tuned.predict(X_val)

print("\n--- Comparison on Validation ---")
print(f"Before tuning: R²=0.9352  MAE=504.84")
print(f"After tuning:  R²={r2_score(y_val, rf_tuned_pred):.4f}  MAE={mean_absolute_error(y_val, rf_tuned_pred):.2f}")

if r2_score(y_val, rf_tuned_pred) > 0.9352:
    print("\n✅ Tuned model is better → use tuned!")
else:
    print("\n❌ No improvement → keep original model!")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
Best params: {'n_estimators': 300, 'min_samples_split': 5, 'max_depth': 20}

--- Comparison on Validation ---
Before tuning: R²=0.9352  MAE=504.84
After tuning:  R²=0.9150  MAE=624.42

❌ No improvement → keep original model!


In [50]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Random Forest — Before Tuning
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("--- Random Forest Before Tuning ---")
print(f"R²:  {r2_score(y_val, rf_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_val, rf_pred):.2f}")

--- Random Forest Before Tuning ---
R²:  0.9352
MAE: 504.84


In [51]:
# ─────────────────────────────────────
# Compare Train vs Validation
# ─────────────────────────────────────

# Predict on BOTH train and validation
rf_pred_train = rf.predict(X_train)
rf_pred_val   = rf.predict(X_val)

print("--- Random Forest: Train vs Validation ---")
print(f"{'':20} {'R²':>10} {'MAE':>10}")
print(f"{'Train':20} {r2_score(y_train, rf_pred_train):>10.4f} {mean_absolute_error(y_train, rf_pred_train):>10.2f}")
print(f"{'Validation':20} {r2_score(y_val, rf_pred_val):>10.4f} {mean_absolute_error(y_val, rf_pred_val):>10.2f}")

--- Random Forest: Train vs Validation ---
                             R²        MAE
Train                    0.9905     195.23
Validation               0.9352     504.84


In [55]:
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

# ─────────────────────────────────────
# Evaluate ALL models on Validation Set
# ─────────────────────────────────────

results = []

# 1. Baseline
baseline_val = np.full(len(y_val), np.mean(y_train))
results.append({
    'Model':  'Baseline (mean)',
    'R² Train': '-',
    'R² Val':  f"{r2_score(y_val, baseline_val):.4f}",
    'MAE Train': '-',
    'MAE Val':  f"{mean_absolute_error(y_val, baseline_val):.2f}"
})

# 2. Linear Regression
lr_train_pred = lr.predict(X_train)
lr_val_pred   = lr.predict(X_val)
results.append({
    'Model':   'Linear Regression',
    'R² Train': f"{r2_score(y_train, lr_train_pred):.4f}",
    'R² Val':   f"{r2_score(y_val, lr_val_pred):.4f}",
    'MAE Train':f"{mean_absolute_error(y_train, lr_train_pred):.2f}",
    'MAE Val':  f"{mean_absolute_error(y_val, lr_val_pred):.2f}"
})

# 3. Random Forest (default)
rf_train_pred = rf.predict(X_train)
rf_val_pred   = rf.predict(X_val)
results.append({
    'Model':   'Random Forest (default)',
    'R² Train': f"{r2_score(y_train, rf_train_pred):.4f}",
    'R² Val':   f"{r2_score(y_val, rf_val_pred):.4f}",
    'MAE Train':f"{mean_absolute_error(y_train, rf_train_pred):.2f}",
    'MAE Val':  f"{mean_absolute_error(y_val, rf_val_pred):.2f}"
})

# 4. Random Forest (tuned)
rf_tuned_train_pred = rf_tuned.predict(X_train)
rf_tuned_val_pred   = rf_tuned.predict(X_val)
results.append({
    'Model':   'Random Forest (tuned)',
    'R² Train': f"{r2_score(y_train, rf_tuned_train_pred):.4f}",
    'R² Val':   f"{r2_score(y_val, rf_tuned_val_pred):.4f}",
    'MAE Train':f"{mean_absolute_error(y_train, rf_tuned_train_pred):.2f}",
    'MAE Val':  f"{mean_absolute_error(y_val, rf_tuned_val_pred):.2f}"
})

# ─────────────────────────────────────
# Print Table
# ─────────────────────────────────────
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

                  Model R² Train  R² Val MAE Train MAE Val
        Baseline (mean)        - -0.0000         - 2294.77
      Linear Regression   0.7298  0.7291   1154.58 1156.53
Random Forest (default)   0.9905  0.9352    195.23  504.84
  Random Forest (tuned)   0.9492  0.9150    495.78  624.42


In [56]:
# Use RF Default for Final Evaluation
rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

# Final Evaluation on TEST SET
rf_test_pred = rf_final.predict(X_test)

print("--- 📍 Step 12: Final Evaluation on Test Set ---")
print(f"R²:  {r2_score(y_test, rf_test_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, rf_test_pred):.2f}")

--- 📍 Step 12: Final Evaluation on Test Set ---
R²:  0.9331
MAE: 502.75
